# 의도 분류 + 검색 성능 평가 (혜림 요청 11/23) — Colab · Qwen3.5-27B · B 검색

질문이 **① 의도 분류**(규칙 + 분류 보조 LLM, backend 그대로)를 거쳐 실제로 **검색까지 가는지**, 가면 **B 검색이 정답 조문을 가져오는지**를 같이 봐요. 답변 생성은 안 해요.

```
질문 → guard(마스킹) → classify (보안문서 판단 + 의도 분류, backend main 그대로 · 분류 LLM = Qwen3.5-27B)
     → 검색으로 가나?  backend app/graph/build.py 의 run_pipeline 분기와 같은 순서
          sensitive · 대화기록 거절 · 기밀문서(conf_doc) · refuse · no_question · out_of_scope · lookup · data → 검색 안 함
          rule · both → B 검색 (조문 후보 40조각 → 항 리랭크 → LLM에 넘길 조문 5개)
```

| 데이터 | 정답 조각 있는 문항 | 정답 |
| --- | --- | --- |
| 임직원 `eval/golden/golden_set.json` | 270 = dev 75 · test 166 · uncertain 29 (uncertain은 점수에서 빼고 따로) | `gold_chunks.primary`(조문) · `gold_hang.primary`(항) |
| 인사팀 `eval/golden/admin_golden_set.json` | 93 | `gold_chunks.primary`(조문)만 |

**지표** (정답 조각이 있는 문항 기준)

| 지표 | 뜻 |
| --- | --- |
| 검색 누락률 | 정답 조각이 있는데 분류 때문에 검색을 안 한 문항 ÷ 정답 조각이 있는 문항 |
| **2단계 종합 Recall@5** | 검색 안 한 문항은 0, 검색한 문항은 상위 조문 5개 Recall. 사용자가 실제로 받는 검색 품질 |
| 분류 없이 검색만 Recall@5 | 같은 문항을 분류를 건너뛰고 모두 검색했을 때. **이것 − 2단계 종합 = 분류 때문에 잃은 만큼** |
| 검색한 문항만 | 조문 Recall@5 · @10 · @40(후보), MRR, nDCG@5, 정답 항 전달률(임직원만) |

- 조문 지표는 `gold_chunks.primary`를 DB `data/hang/old_to_new.json`으로 조문(`article_id`)으로 바꿔서 B 검색의 조문 순서와 비교해요. Recall@40(후보) = ② 후보 40조각이 속한 조문 전체에 정답 조문이 있는 비율이에요.
- 정답 항 전달률 = `gold_hang.primary` 항의 앞 30자·끝 30자가 LLM에 넘기는 조문 5개 글 안에 다 있는 비율 (b_search_eval.ipynb와 같은 방식). `gold_hang.auto`가 `false`인 문항은 따로 내요.
- 인사팀은 검색어 = 질문 + `search_hint`(분류가 집계 도구를 고르면 붙음). 그래서 인사팀의 '분류 없이 검색만'은 힌트 없이 질문만이에요. 힌트 효과는 참고 지표로 따로 내요.

**검색은 B 방식** (11/19 최종 선정): RAG 브랜치 `feat/b-search-notebook`(04048ce)의 `colab/b_search_eval.ipynb`와 같은 설정·코드예요. 재검색(근거 점수 낮을 때 1회)은 안 보고 첫 검색만 봐요.

**GPU 하나를 나눠 쓰는 순서**: ① 27B 다운로드를 백그라운드로 걸어 두고 그 사이 B 검색 모델로 '분류 없이 검색만'을 먼저 다 돌려요 → ② 검색 모델을 내리고 vLLM으로 분류 → ③ vLLM을 끄고 인사팀 `search_hint` 붙은 검색만 더 돌려요. 검색 속도는 ①에서 GPU를 혼자 쓸 때 재요.

**주의**
- 분류 LLM 응답이 실패하면 backend가 규칙 결과로 대체해요. `chat_json`을 감싸서 실패 횟수를 세요 (요청 실패·JSON 아님 + 단계별로 규칙 대체된 횟수).
- 임직원 분류 LLM은 기준일로 `date.today()`를 써요 (인사팀 분류도 같음). **코드는 안 고쳤고**, 실행일을 `summary.json`의 `classify_today`에 남겨요.
- 징계·노사 질문 중 일부는 기밀문서(`conf_doc`)로 분류돼 규정 검색을 안 해요 (의도된 동작). 누락 목록에서 이유를 `기밀문서`로 따로 표시해요.

**실행**: 런타임 → 런타임 유형 변경 → **A100(40GB↑) 또는 H100** → 모두 실행. 첫 실행은 27B 다운로드 때문에 1시간쯤 걸려요. 끝나면 zip이 내려받아지고 Drive가 연결되면 `MyDrive/hr_chatbot_eval/`에도 복사돼요.

## 1. 설정 · 저장소 받기

In [ ]:
import os, sys, json, re, time, gc, hashlib, subprocess, shutil, importlib.util
from pathlib import Path
from datetime import date, datetime
from collections import defaultdict, Counter

ROOT = Path("/content/hr_chatbot")
ROOT.mkdir(parents=True, exist_ok=True)
OUT_DIR = Path("/content/intent_search_results") / f"intent_search_{datetime.now():%Y%m%d_%H%M}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    "branches": {"backend": "main", "DB": "main", "RAG": "main"},
    # 분류 보조 LLM (backend 분류 코드가 부름)
    "llm_model": "auto",            # auto: 75GB↑ Qwen/Qwen3.5-27B · 39GB↑ -FP8. 27B가 안 들어가면 멈춤
    "served_name": "qwen3.5-27b",
    "llm_port": 8000,
    "max_model_len": 8192,
    "admin_login": "E1307",         # 인사팀 골든셋을 만들 때 쓴 계정 (backend scripts/eval_admin_intent.py)
    "employee_login": "E1584",      # 페르소나 없는 문항용 (backend scripts/eval_intent.py)
    # B 검색 (RAG feat/b-search-notebook 04048ce b_search_eval.ipynb 와 같음)
    "embed_model": "BAAI/bge-m3", "rerank_model": "BAAI/bge-reranker-v2-m3",
    "k_dense": 30, "k_bm25": 30, "k_candidates": 40, "rrf_c": 60,
    "bm25_weight": 1.0, "dense_weight": 1.0, "law_weight": 0.6, "rewrite_bm25": True,
    "internal_bonus": 0.0, "k_context": 5, "attach_appendix": True, "max_chars_per_article": 1200,
}

COMMITS = {}
for repo, branch in CFG["branches"].items():
    d = ROOT / repo
    if d.exists():
        shutil.rmtree(d)
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", branch,
                    f"https://github.com/Jaeyong-s-HR-Chatbot/{repo}.git", str(d)], check=True)
    COMMITS[repo] = subprocess.run(["git", "-C", str(d), "log", "-1", "--oneline"],
                                   capture_output=True, text=True).stdout.strip()
    print(f"{repo:8s} {COMMITS[repo]}")
BACKEND, DB, RAG = ROOT / "backend", ROOT / "DB", ROOT / "RAG"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "sentence-transformers>=3.4", "rank_bm25", "kiwipiepy",
                "python-dotenv", "httpx", "PyJWT", "openai"], check=True)

## 2. GPU 확인 · 분류 모델 고르기

In [ ]:
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
if not gpu:
    raise RuntimeError("GPU가 없어요. 런타임 → 런타임 유형 변경 → A100 또는 H100으로 바꾸고 다시 실행하세요.")
lines = gpu.splitlines()
GPU_NAME, N_GPU = lines[0].split(",")[0].strip(), len(lines)
TOTAL_GB = sum(int(l.split(",")[1]) for l in lines) / 1024
if CFG["llm_model"] == "auto":
    if TOTAL_GB >= 75:   CFG["llm_model"] = "Qwen/Qwen3.5-27B"
    elif TOTAL_GB >= 39: CFG["llm_model"] = "Qwen/Qwen3.5-27B-FP8"
    else:
        raise RuntimeError(f"{GPU_NAME} {TOTAL_GB:.0f}GB로는 Qwen3.5-27B가 안 들어가요 (L4 등). A100(40GB↑) 또는 H100으로 바꾸세요.")
print(f"GPU {GPU_NAME} × {N_GPU}, {TOTAL_GB:.0f}GB → 분류 LLM {CFG['llm_model']}")

## 3. vLLM 설치 · 27B 다운로드 (백그라운드)

GPU는 안 써요. 시작해 두고 아래 셀을 계속 실행하세요 (그 사이 B 검색을 먼저 돌려요).

In [ ]:
VLLM_ENV = Path("/content/vllm-env")
SETUP_LOG = Path("/content/llm_setup.log")
HF_TOKEN = ""
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN") or ""      # 보안 비밀(열쇠 아이콘)에 있으면 다운로드가 빨라요. 없어도 됨
except Exception:
    pass
setup_script = f"""set -e
echo "[1/2] vLLM 설치 $(date)"
if [ ! -x {VLLM_ENV}/bin/vllm ]; then
  pip install -q uv
  rm -rf {VLLM_ENV}
  uv venv {VLLM_ENV} --python 3.12 -q
  uv pip install --python {VLLM_ENV}/bin/python -q "vllm>=0.17.0" --torch-backend=auto
fi
echo "[2/2] 모델 다운로드 {CFG['llm_model']} $(date)"
{"export HF_TOKEN=" + HF_TOKEN if HF_TOKEN else ""}
{VLLM_ENV}/bin/python -c "from huggingface_hub import snapshot_download; snapshot_download('{CFG['llm_model']}')"
echo "SETUP_DONE $(date)"
"""
Path("/content/llm_setup.sh").write_text(setup_script)
if "SETUP_DONE" not in (SETUP_LOG.read_text() if SETUP_LOG.exists() else ""):
    SETUP_PROC = subprocess.Popen(f"bash /content/llm_setup.sh > {SETUP_LOG} 2>&1", shell=True)
    print("백그라운드 시작. 진행 상황:", SETUP_LOG)
else:
    print("이미 설치·다운로드됨")

## 4. backend 분류 코드 · 골든셋 불러오기

backend를 DB 없이 불러요 (분류·마스킹만 씀). 분류 LLM은 7번에서 vLLM에 연결해요.

In [ ]:
os.environ.update({"HRDB_URL": "", "CHAT_LOG_DB": "false", "LLM_BASE_URL": "", "CLASSIFY_LLM": "off",
                   "SEARCH_MODE": "keyword", "PSEUDO_KEY": hashlib.sha256(os.urandom(16)).hexdigest()})
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))
from app import config as be_config
from app.graph import nodes as be_nodes, llm_classify, admin_intent_llm, admin_params_llm, conf_doc
from app.graph import retriever as be_retriever          # 질의 재작성(glossary) = B 검색의 BM25 검색어 보강

# 골든셋 · dev/test 분할 (팀 채점 스크립트 그대로)
os.environ["BACKEND_DIR"] = str(BACKEND)
spec = importlib.util.spec_from_file_location("eval_golden", RAG / "scripts/eval_golden.py")
eg = importlib.util.module_from_spec(spec); spec.loader.exec_module(eg)
EMP_RAW = eg.load_items()
SPLIT = eg.get_splits(EMP_RAW)
ADMIN_RAW = json.loads((RAG / "eval/golden/admin_golden_set.json").read_text(encoding="utf-8"))["items"]
PERSONA = {p["golden_persona"]: p for p in json.loads((DB / "data/meta/golden_personas.json").read_text(encoding="utf-8"))}
O2N = json.loads((DB / "data/hang/old_to_new.json").read_text(encoding="utf-8"))   # 조문 조각 → 조문(article_id)


def to_articles(chunk_ids):
    return list(dict.fromkeys(O2N[c]["article_id"] for c in chunk_ids if c in O2N))


ITEMS = []
for it in EMP_RAW:
    p = PERSONA.get(it.get("persona"), {})
    gh = it.get("gold_hang") or {}
    ITEMS.append({"set": "임직원", "id": it["id"], "split": SPLIT[it["id"]], "uncertain": bool(it["uncertain"]),
                  "question": it["question"], "role": "employee",
                  "login": it.get("emp_id") or p.get("emp_id") or CFG["employee_login"], "own_name": p.get("name", ""),
                  "gold_chunks": it["gold_chunks"]["primary"], "gold_articles": to_articles(it["gold_chunks"]["primary"]),
                  "gold_hang": gh.get("primary") or [], "hang_auto": gh.get("auto")})
for it in ADMIN_RAW:
    ITEMS.append({"set": "인사팀", "id": it["id"], "split": it["split"], "uncertain": bool(it["uncertain"]),
                  "question": it["question"], "role": it["role"],
                  "login": CFG["admin_login"] if it["role"] == "hr_admin" else CFG["employee_login"], "own_name": "",
                  "gold_chunks": it["gold_chunks"]["primary"], "gold_articles": to_articles(it["gold_chunks"]["primary"]),
                  "gold_hang": [], "hang_auto": None})
BY_ID = {x["id"]: x for x in ITEMS}
missing = [(x["id"], c) for x in ITEMS for c in x["gold_chunks"] if c not in O2N]
assert not missing, f"old_to_new.json에 없는 정답 조각: {missing[:5]}"
print(Counter((x["set"], x["split"], bool(x["gold_chunks"])) for x in ITEMS))

## 5. B 검색 준비 (b_search_eval.ipynb 2~4절과 같은 코드)
- **조문 조각** (`DB/data/regs`·`laws/chunks.jsonl`): ② 검색 대상 (BM25 + BGE-M3)
- **항 조각** (`DB/data/hang/chunks.jsonl`): ③ 리랭커가 점수를 매기는 단위

In [ ]:
import numpy as np, pandas as pd, torch


def jl(p): return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]


def _cite(r):
    art, t = (r.get("article") or "").strip(), (r.get("article_title") or "").strip()
    return f"{r['doc_name']} {art}" + (f"({t[:20]})" if t and art.startswith("제") else "")


def load_corpus(paths):
    out = []
    for p in paths:
        for r in jl(p):
            if not r.get("searchable"):
                continue
            out.append({"id": r["chunk_id"], "doc_id": r["document_id"], "doc_name": r["doc_name"],
                        "doc_kind": r.get("doc_kind") or ("법령" if r["chunk_id"].startswith("L") else "내부규정"),
                        "article": (r.get("article") or "").strip(), "article_title": (r.get("article_title") or "").strip(),
                        "part": r.get("part") or 1, "citation": _cite(r), "revised": r.get("effective_date") or "",
                        "text": r["text"], "article_id": r.get("article_id"),
                        "embed_text": f"{r['doc_name']} {r.get('article') or ''} {r.get('article_title') or ''}".strip() + "\n" + r["text"]})
    return out


OLDC = load_corpus([DB / "data/regs/chunks.jsonl", DB / "data/laws/chunks.jsonl"])   # 조문 조각
NEWC = load_corpus([DB / "data/hang/chunks.jsonl"])                                   # 항 조각
ART = {k: v["article_id"] for k, v in O2N.items()} | {c["id"]: c["article_id"] for c in NEWC}   # 조각 → 조문
ARTICLES = {a["article_id"]: a for a in jl(DB / "data/hang/articles.jsonl")}                    # 조문 원문
NEW_ID = {c["id"]: c for c in NEWC}
NEW_BY_ART = defaultdict(list)          # 조문 → 항 조각 번호 (조문 안 순서)
for j, c in enumerate(NEWC):
    NEW_BY_ART[c["article_id"]].append(j)
OLD_ARTICLE = defaultdict(list)         # (문서, 조문) → 조문 조각 번호 (part 순). 별표 찾기에 씀
for i, c in enumerate(OLDC):
    if c["article"]:
        OLD_ARTICLE[(c["doc_id"], c["article"])].append(i)
for k in OLD_ARTICLE:
    OLD_ARTICLE[k].sort(key=lambda i: OLDC[i]["part"])
CITE_OF_ART = {}
for c in NEWC:
    CITE_OF_ART.setdefault(c["article_id"], c["citation"])
print(f"조문 조각 {len(OLDC):,} · 항 조각 {len(NEWC):,} · 조문 {len(NEW_BY_ART):,}")

In [ ]:
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"


def _load(cls, name, **kw):
    try:
        return cls(name, device=DEVICE, model_kwargs={"torch_dtype": torch.float16 if HAS_GPU else torch.float32}, **kw)
    except TypeError:
        m = cls(name, device=DEVICE, **kw)
        if HAS_GPU:
            m.model.half() if hasattr(m, "model") else m.half()
        return m


def load_search_models():
    global EMBEDDER, RERANKER
    EMBEDDER = _load(SentenceTransformer, CFG["embed_model"])
    EMBEDDER.max_seq_length = 1024
    RERANKER = _load(CrossEncoder, CFG["rerank_model"], max_length=1024)


def free_search_models():
    # vLLM이 GPU를 다 쓰도록 검색 모델을 내려요 (색인 DOC_EMB는 CPU 배열이라 그대로 둠)
    global EMBEDDER, RERANKER
    EMBEDDER = RERANKER = None
    gc.collect()
    if HAS_GPU:
        torch.cuda.empty_cache()


kiwi = Kiwi()
_KEEP = {"NNG", "NNP", "NR", "SL", "SN", "SH", "XR", "VV", "VA"}
def ko_tokens(text):
    text = re.sub(r"\[[^\]]*\]", " ", text)           # [전화] 같은 마스킹 표시는 뺌
    return [t.form + ("다" if t.tag in ("VV", "VA") else "") for t in kiwi.tokenize(text) if t.tag in _KEEP]


load_search_models()
t0 = time.time()
BM25 = BM25Okapi([ko_tokens(c["embed_text"]) for c in OLDC])
DOC_EMB = EMBEDDER.encode([c["embed_text"] for c in OLDC], batch_size=32, normalize_embeddings=True,
                          convert_to_numpy=True, show_progress_bar=True).astype(np.float32)
print(f"BM25 + 임베딩 {time.time() - t0:.0f}초 · {DOC_EMB.shape} · GPU {torch.cuda.get_device_name(0) if HAS_GPU else 'cpu'}")

In [ ]:
_QCACHE, _RCACHE = {}, {}
RR = {"pairs": 0, "sec": 0.0}


def embed_query(text):
    if text not in _QCACHE:
        _QCACHE[text] = EMBEDDER.encode([re.sub(r"\[[^\]]*\]", " ", text)], normalize_embeddings=True,
                                        convert_to_numpy=True)[0].astype(np.float32)
    return _QCACHE[text]


def bm25_rank(query, n):
    s = BM25.get_scores(ko_tokens(query))
    return [int(i) for i in np.argsort(-s)[:n] if s[i] > 0]


def dense_rank(query, n):
    return [int(i) for i in np.argsort(-(DOC_EMB @ embed_query(query)))[:n]]


def candidates(question):
    # ② 가중 RRF → 조문 조각 후보 40개. 법령 조각은 law_weight 배
    n = CFG["k_candidates"]
    bq = be_retriever.rewrite(question) if CFG["rewrite_bm25"] else question
    lists = [(bm25_rank(bq, max(CFG["k_bm25"], n)), CFG["bm25_weight"]),
             (dense_rank(question, max(CFG["k_dense"], n)), CFG["dense_weight"])]
    score = defaultdict(float)
    for lst, w in lists:
        for r, i in enumerate(lst):
            score[i] += w / (CFG["rrf_c"] + r + 1)
    for i in score:
        if OLDC[i]["doc_kind"] == "법령":
            score[i] *= CFG["law_weight"]
    return sorted(score, key=score.get, reverse=True)[:n]


def rerank_hang(question, idxs, batch_size=16):
    q = re.sub(r"\[[^\]]*\]", " ", question)
    todo = [j for j in idxs if (q, j) not in _RCACHE]
    if todo:
        t = time.perf_counter()
        s = np.asarray(RERANKER.predict([(q, NEWC[j]["embed_text"]) for j in todo], batch_size=batch_size,
                                        convert_to_numpy=True, show_progress_bar=False), dtype=float).reshape(-1)
        if s.min() < 0 or s.max() > 1:
            s = 1 / (1 + np.exp(-s))
        _RCACHE.update({(q, j): float(v) for j, v in zip(todo, s)})
        RR["pairs"] += len(todo)
        RR["sec"] += time.perf_counter() - t
    b = CFG["internal_bonus"]
    return {j: min(1.0, _RCACHE[(q, j)] + (b if NEWC[j]["doc_kind"] == "내부규정" else 0.0)) for j in idxs}


def search_B(question):
    # → (조문 순서 = 후보 40조각이 속한 조문 전체를 리랭커 점수로 정렬, {항 번호: 점수})
    arts = list(dict.fromkeys(ART[OLDC[i]["id"]] for i in candidates(question) if OLDC[i]["id"] in ART))
    arts = [a for a in arts if NEW_BY_ART.get(a)]
    sc = rerank_hang(question, [j for a in arts for j in NEW_BY_ART[a]])
    art_score = {a: max(sc[j] for j in NEW_BY_ART[a]) for a in arts}
    return sorted(arts, key=lambda a: -art_score[a]), sc


def _head(c):
    return f"{c['article']}({c['article_title']})" if c["article_title"] else c["article"]


def context_B(art_rank, sc, k=None):
    # LLM에 넘길 조문 5개: 조문마다 점수 높은 항부터 이어 붙임 + 별표 1개 → 1,200자
    out, cap = [], CFG["max_chars_per_article"]
    for a in art_rank[:k or CFG["k_context"]]:
        js = sorted(NEW_BY_ART[a], key=lambda j: -sc[j])
        parts = []
        for n, j in enumerate(js):
            t, h = NEWC[j]["text"], _head(NEWC[j])
            parts.append(t[len(h):].lstrip() if n and h and t.startswith(h) else t)
        text, c0 = "\n".join(parts), NEWC[js[0]]
        if CFG["attach_appendix"] and not c0["article"].startswith("별표"):
            for no in dict.fromkeys(re.findall(r"별표\s*(\d+)", ARTICLES.get(a, {}).get("text", text))):
                appx = OLD_ARTICLE.get((c0["doc_id"], f"별표 {no}"))
                if appx and ART.get(OLDC[appx[0]]["id"]) != a:
                    text += f"\n[참조 별표 {no}]\n" + "\n".join(OLDC[x]["text"] for x in appx)
                    break
        out.append({"id": a, "citation": c0["citation"], "revised": c0["revised"], "doc_kind": c0["doc_kind"],
                    "hang": [NEWC[j]["id"] for j in js], "score": round(sc[js[0]], 4),
                    "text": text if len(text) <= cap else text[:cap] + " …(생략)"})
    return out


def sync():
    if HAS_GPU:
        torch.cuda.synchronize()


def run_search(query):
    # 캐시를 비우고 처음부터 (질문 임베딩 → ② → ③ → 넘길 글). 속도는 이 값으로 재요
    _QCACHE.clear(); _RCACHE.clear(); RR.update(pairs=0, sec=0.0)
    sync(); t = time.perf_counter()
    arts, sc = search_B(query)
    ctx = context_B(arts, sc)
    sync()
    return {"arts": arts, "ctx": ctx, "ms": (time.perf_counter() - t) * 1000}

In [ ]:
def squash(s): return re.sub(r"\s+", "", s)


def hang_sig(cid):
    c = NEW_ID[cid]
    body, h = c["text"], _head(c)
    s = squash(body[len(h):] if h and body.startswith(h) else body)
    return (s[:30], s[-30:]) if len(s) > 30 else (s, s)


def delivered(cid, blob):
    a, b = hang_sig(cid)
    return a in blob and b in blob


def art_metrics(gold_arts, arts):
    # 조문 단위: Recall@5·@10·@40(후보 조문 전체), MRR, nDCG@5
    g = set(gold_arts)
    pos = [i for i, x in enumerate(arts) if x in g]
    dcg = sum(1 / np.log2(i + 2) for i in pos if i < 5)
    idcg = sum(1 / np.log2(i + 2) for i in range(min(len(g), 5)))
    return {"r5": sum(i < 5 for i in pos) / len(g), "r10": sum(i < 10 for i in pos) / len(g),
            "r40": len(pos) / len(g), "mrr": 1 / (pos[0] + 1) if pos else 0.0, "ndcg5": dcg / idcg if idcg else 0.0}


def score_search(x, res):
    m = art_metrics(x["gold_articles"], res["arts"])
    blob = squash("\n".join(h["text"] for h in res["ctx"]))
    m["hang"] = float(np.mean([delivered(c, blob) for c in x["gold_hang"]])) if x["gold_hang"] else None
    m["top5"] = [h["citation"] for h in res["ctx"]]
    m["ms"] = round(res["ms"], 1)
    return m

## 6. ① 분류 없이 검색만 (정답 조각 있는 문항 전부, 질문 그대로)

임직원은 실제 파이프라인도 검색어가 질문 그대로라 이 결과를 그대로 써요. 문항마다 캐시를 비우고 처음부터 재요 (GPU 첫 호출은 예열로 뺌).

In [ ]:
run_search("연차는 며칠이야?")       # 예열
BASE = {}
t0 = time.time()
for x in ITEMS:
    if x["gold_chunks"]:
        BASE[x["id"]] = score_search(x, run_search(x["question"]))
print(f"{len(BASE)}문항 {time.time() - t0:.0f}초 · 검색 p50 {np.median([m['ms'] for m in BASE.values()]):.0f}ms")
(OUT_DIR / "search_base.json").write_text(json.dumps(BASE, ensure_ascii=False), encoding="utf-8")
free_search_models()
print("검색 모델 내림 → vLLM 띄울 준비")

## 7. vLLM 띄우기 (27B 다운로드가 끝날 때까지 기다려요)

In [ ]:
import urllib.request
LLM_LOG = Path("/content/vllm.log")
LLM_API_KEY = hashlib.sha256(os.urandom(16)).hexdigest()[:32]
LLM_BASE_URL = f"http://127.0.0.1:{CFG['llm_port']}/v1"

while "SETUP_DONE" not in SETUP_LOG.read_text():
    if "SETUP_PROC" in globals() and SETUP_PROC.poll() not in (None, 0):
        print(SETUP_LOG.read_text()[-3000:]); raise RuntimeError("vLLM 설치·다운로드 실패 (로그 위)")
    last = (SETUP_LOG.read_text().strip().splitlines() or ["시작 전"])[-1]
    print("설치·다운로드 기다리는 중…", last[:100]); time.sleep(30)

subprocess.run("pkill -f 'vllm serve' || true", shell=True)
time.sleep(3)
help_text = subprocess.run([str(VLLM_ENV / "bin/vllm"), "serve", "--help=all"], capture_output=True, text=True).stdout
cmd = [str(VLLM_ENV / "bin/vllm"), "serve", CFG["llm_model"], "--served-model-name", CFG["served_name"],
       "--host", "127.0.0.1", "--port", str(CFG["llm_port"]), "--api-key", LLM_API_KEY,
       "--max-model-len", str(CFG["max_model_len"]), "--gpu-memory-utilization", "0.9",
       "--max-num-seqs", "16",           # Qwen3.5 Mamba 캐시 때문에 기본 256이면 A100 40GB에서 시작이 안 돼요
       "--reasoning-parser", "qwen3", "--trust-remote-code", "--limit-mm-per-prompt", json.dumps({"image": 0, "video": 0})]
if "--default-chat-template-kwargs" in help_text:
    cmd += ["--default-chat-template-kwargs", json.dumps({"enable_thinking": False})]
if N_GPU > 1:
    cmd += ["--tensor-parallel-size", str(N_GPU)]
print(" ".join(cmd).replace(LLM_API_KEY, "***"))
LLM_PROC = subprocess.Popen(cmd, stdout=open(LLM_LOG, "w"), stderr=subprocess.STDOUT,
                            env={**os.environ, **({"HF_TOKEN": HF_TOKEN} if HF_TOKEN else {})})
t0 = time.time()
while True:
    try:
        req = urllib.request.Request(f"{LLM_BASE_URL}/models", headers={"Authorization": f"Bearer {LLM_API_KEY}"})
        json.loads(urllib.request.urlopen(req, timeout=5).read())
        print(f"✅ vLLM 준비 완료 ({time.time() - t0:.0f}초)"); break
    except Exception:
        if LLM_PROC.poll() is not None:
            log = LLM_LOG.read_text(errors="replace").splitlines()
            keys = ("out of memory", "OutOfMemory", "Error:", "ValueError", "not supported", "No available memory", "max_model_len", "CUDA")
            print("── 원인 후보 ──"); print("\n".join([l for l in log if any(k in l for k in keys)][-25:]) or "\n".join(log[-40:]))
            raise RuntimeError(f"vLLM이 시작하지 못했어요 (전체 로그: {LLM_LOG})")
        time.sleep(15)

## 8. backend 분류 LLM을 vLLM에 연결 · 실패 세기

backend는 분류 LLM을 Ollama 형식 `llm_classify.chat_json`으로 불러요. 이 함수 하나만 vLLM(OpenAI 형식)으로 바꾸고 **프롬프트·스키마·응답 검증·규칙 대체는 backend 그대로**예요 (admin_intent_eval_27b.ipynb와 같은 방식).
- `chat_json` 실패: 요청 오류 + 응답이 JSON이 아님
- 단계별 규칙 대체: 임직원 의도(`llm_classify.classify`), 인사팀 의도(`admin_intent_llm.classify`), 인사팀 인자(`admin_params_llm.extract`), 보안문서(`conf_doc._llm_detect`)가 `None`을 돌려준 횟수 (스키마는 맞는데 값이 틀린 경우 포함)

In [ ]:
CALLS = {"n": 0, "fail": 0, "bad_json": 0, "ms": []}
FALLBACK = Counter()      # 단계별 규칙 대체 횟수
CUR = {"calls": 0, "fail": 0, "fallback": []}     # 지금 분류 중인 문항 하나의 기록


def _vllm_chat_json(system, user, schema, num_predict=256):
    from openai import OpenAI
    global _LLM
    if "_LLM" not in globals():
        _LLM = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY, timeout=300, max_retries=2)
    r = _LLM.chat.completions.create(
        model=CFG["served_name"],
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
        temperature=0, max_tokens=num_predict,
        response_format={"type": "json_schema", "json_schema": {"name": "out", "schema": schema}},
        extra_body={"chat_template_kwargs": {"enable_thinking": False}})
    return be_nodes._clean(r.choices[0].message.content or "")


def counted_chat_json(transport):
    # chat_json을 감싸서 호출·실패를 셈. 실패하면 backend가 잡아 규칙 결과로 대체하도록 ValueError를 올림
    def chat_json(system, user, schema, num_predict=256):
        CALLS["n"] += 1; CUR["calls"] += 1
        t = time.perf_counter()
        try:
            out = transport(system, user, schema, num_predict)
        except Exception:
            CALLS["fail"] += 1; CUR["fail"] += 1
            raise ValueError("분류 LLM 요청 실패")
        finally:
            CALLS["ms"].append((time.perf_counter() - t) * 1000)
        try:
            json.loads(out)
        except (json.JSONDecodeError, TypeError):
            CALLS["bad_json"] += 1; CUR["fail"] += 1
        return out
    return chat_json


def count_fallback(module, name, label):
    orig = getattr(module, name)
    if getattr(orig, "_counted", False):      # 셀을 다시 실행해도 두 번 감싸지 않게
        return
    def wrapped(*a, **kw):
        r = orig(*a, **kw)
        if r is None:
            FALLBACK[label] += 1; CUR["fallback"].append(label)
        return r
    wrapped._counted = True
    setattr(module, name, wrapped)


def connect_llm(transport):
    llm_classify.chat_json = counted_chat_json(transport)
    be_config.CLASSIFY_LLM = True           # 분류 보조 LLM 켜기 (enabled() = CLASSIFY_LLM and not USE_FAKE_LLM)
    be_config.USE_FAKE_LLM = False
    be_config.CLASSIFY_MODEL = CFG["served_name"]
    assert llm_classify.enabled() and llm_classify.is_local()   # 보안문서 LLM 판단은 로컬 LLM일 때만 켜져요
    count_fallback(llm_classify, "classify", "임직원 의도")
    count_fallback(admin_intent_llm, "classify", "인사팀 의도")
    count_fallback(admin_params_llm, "extract", "인사팀 인자")
    count_fallback(conf_doc, "_llm_detect", "보안문서")

In [ ]:
connect_llm(_vllm_chat_json)
for q, role in (("결혼하면 휴가 며칠이야?", "employee"), ("올해 연차 사용률 50% 안 되는 부서 어디야?", "hr_admin")):
    s = {"employee_id": CFG["admin_login"], "role": role, "question": q, "own_name": ""}
    s |= be_nodes.guard(s); t = time.time(); s |= be_nodes.classify(s)
    print(f"{time.time() - t:5.1f}초 | {role:8s} | {q} → {s['intent']} {s.get('admin_tool') or ''} {s.get('search_hint') or ''}")
print(CALLS["n"], "호출,", CALLS["fail"], "실패")

## 9. 분류 → 검색으로 가나? (전 문항)

분기는 backend `app/graph/build.py`의 `run_pipeline`과 같은 순서예요. 정답 조각이 없는 문항도 분류해요 (불필요한 검색 세기).

In [ ]:
def route(s):
    # build.run_pipeline 분기 그대로 → (검색하나, 이유)
    if s["intent"] == "sensitive":
        return False, "sensitive"
    if s.get("refuse_reason") == "chat_log":
        return False, "refuse(대화 기록)"
    if s.get("conf_doc"):
        return False, "기밀문서"
    if s["intent"] == "refuse":
        return False, "refuse"
    if s["intent"] == "no_question" or not be_nodes.has_question_content(s["masked_question"]):
        return False, "no_question"
    if s["intent"] == "out_of_scope":
        return False, "out_of_scope"
    if s["intent"] in ("lookup", "data"):
        return False, s["intent"]
    return True, s["intent"]


def classify_item(x):
    s = {"employee_id": x["login"], "role": x["role"], "question": x["question"], "own_name": x["own_name"]}
    CUR.update(calls=0, fail=0, fallback=[])
    t = time.perf_counter()
    s |= be_nodes.guard(s)
    s |= be_nodes.classify(s)
    ms = (time.perf_counter() - t) * 1000
    searched, reason = route(s)
    hint = s.get("search_hint") or ""
    return {"intent": s["intent"], "conf_doc": s.get("conf_doc"), "admin_tool": s.get("admin_tool"),
            "search_hint": hint, "searched": searched, "reason": reason,
            "query": f"{x['question']} {hint}" if hint else x["question"],      # nodes.retrieve와 같은 검색어
            "classify_ms": round(ms, 1), "llm_calls": CUR["calls"], "llm_fail": CUR["fail"], "fallback": list(CUR["fallback"])}


def classify_all():
    CALLS.update(n=0, fail=0, bad_json=0, ms=[]); FALLBACK.clear()
    out = {}
    t0 = time.time()
    for n, x in enumerate(ITEMS, 1):
        out[x["id"]] = classify_item(x)
        if n % 50 == 0:
            print(f"{n}/{len(ITEMS)} {time.time() - t0:.0f}초")
    return out

In [ ]:
CLASSIFY_TODAY = date.today().isoformat()       # 분류 LLM 기준일 (date.today(), 코드 안 고침)
CLS = classify_all()
(OUT_DIR / "classify.json").write_text(json.dumps(CLS, ensure_ascii=False), encoding="utf-8")
print(f"LLM 호출 {CALLS['n']}번 · 실패 {CALLS['fail']}번(요청 {CALLS['fail'] - CALLS['bad_json']} · JSON 아님 {CALLS['bad_json']}) · 규칙 대체 {dict(FALLBACK)}")
print(Counter((BY_ID[i]["set"], c["reason"]) for i, c in CLS.items()))

## 10. vLLM 끄기 → 인사팀 `search_hint` 붙은 검색만 더 돌리기

In [ ]:
if "LLM_PROC" in globals() and LLM_PROC.poll() is None:
    LLM_PROC.terminate()
    try: LLM_PROC.wait(timeout=60)
    except Exception: LLM_PROC.kill()
subprocess.run("pkill -f 'vllm serve' || true", shell=True)
time.sleep(5)
load_search_models()
run_search("연차는 며칠이야?")       # 예열

In [ ]:
HINTED = {}
for x in ITEMS:
    c = CLS[x["id"]]
    if x["gold_chunks"] and c["search_hint"]:
        HINTED[x["id"]] = score_search(x, run_search(c["query"]))
print(f"search_hint 붙은 검색 {len(HINTED)}문항")

## 11. 문항별 결과 · 요약

- `pipe_*`: 실제 파이프라인 검색 결과 (검색 안 한 문항은 비어 있음)
- `stage2_r5`: 2단계 종합 Recall@5 (검색 안 했으면 0)
- `base_*`: 분류 없이 질문만으로 검색

In [ ]:
def build_rows():
    rows = []
    for x in ITEMS:
        c = CLS[x["id"]]
        r = {k: x[k] for k in ("set", "id", "split", "uncertain", "question", "gold_chunks", "gold_articles", "gold_hang", "hang_auto")}
        r |= {k: c[k] for k in ("intent", "conf_doc", "admin_tool", "search_hint", "searched", "reason", "query",
                                "classify_ms", "llm_calls", "llm_fail", "fallback")}
        r["has_gold"] = bool(x["gold_chunks"])
        if r["has_gold"]:
            base = BASE[x["id"]]
            pipe = HINTED[x["id"]] if c["search_hint"] else base
            r |= {f"base_{k}": v for k, v in base.items()}
            if c["searched"]:
                r |= {f"pipe_{k}": v for k, v in pipe.items()}
            r["stage2_r5"] = pipe["r5"] if c["searched"] else 0.0
            if c["search_hint"]:
                r["hint_on_r5"], r["hint_off_r5"] = pipe["r5"], base["r5"]
        rows.append(r)
    return rows


def _mean(v):
    v = [x for x in v if x is not None]
    return round(float(np.mean(v)), 4) if v else None


def _pct(v, q):
    return round(float(np.percentile(v, q)), 1) if len(v) else None


GROUPS = {   # 이름 → 문항 고르기
    "임직원 dev": lambda r: r["set"] == "임직원" and r["split"] == "dev",
    "임직원 test": lambda r: r["set"] == "임직원" and r["split"] == "test",
    "인사팀": lambda r: r["set"] == "인사팀",
}
EXTRA_GROUPS = {
    "임직원 uncertain (점수 제외, 따로)": lambda r: r["set"] == "임직원" and r["uncertain"],
    "인사팀 (uncertain 6 제외)": lambda r: r["set"] == "인사팀" and not r["uncertain"],
    "인사팀 dev": lambda r: r["set"] == "인사팀" and r["split"] == "dev",
    "인사팀 test": lambda r: r["set"] == "인사팀" and r["split"] == "test",
}


def group_summary(rows):
    g = [r for r in rows if r["has_gold"]]
    s = [r for r in g if r["searched"]]
    miss = [r for r in g if not r["searched"]]
    hang_auto = [r["pipe_hang"] for r in s if r["gold_hang"] and r["hang_auto"] in (True, "checked")]
    hang_rest = [r["pipe_hang"] for r in s if r["gold_hang"] and r["hang_auto"] is False]
    hinted = [r for r in g if r["search_hint"]]
    out = {
        "문항(정답 조각 있음)": len(g),
        "검색 누락": len(miss), "검색 누락률": round(len(miss) / len(g), 4) if g else None,
        "  └ 기밀문서": sum(r["reason"] == "기밀문서" for r in miss),
        "2단계 종합 Recall@5": _mean([r["stage2_r5"] for r in g]),
        "분류 없이 검색만 Recall@5": _mean([r["base_r5"] for r in g]),
        "분류로 잃은 Recall@5": None,
        "검색한 문항": len(s),
        "검색한 문항 Recall@5": _mean([r["pipe_r5"] for r in s]),
        "검색한 문항 Recall@10": _mean([r["pipe_r10"] for r in s]),
        "검색한 문항 Recall@40(후보)": _mean([r["pipe_r40"] for r in s]),
        "검색한 문항 MRR": _mean([r["pipe_mrr"] for r in s]),
        "검색한 문항 nDCG@5": _mean([r["pipe_ndcg5"] for r in s]),
        "정답 항 전달률": _mean(hang_auto), "정답 항 전달률 문항": len(hang_auto),
        "정답 항 전달률 (auto=false, 따로)": _mean(hang_rest), "auto=false 문항": len(hang_rest),
        "분류 없이 검색만 MRR": _mean([r["base_mrr"] for r in g]),
        "분류 없이 검색만 nDCG@5": _mean([r["base_ndcg5"] for r in g]),
        "참고: 정답 조각 없는데 검색": sum(r["searched"] for r in rows if not r["has_gold"]),
        "참고: 정답 조각 없는 문항": sum(not r["has_gold"] for r in rows),
        "참고: search_hint 붙은 문항": len(hinted),
    }
    if out["2단계 종합 Recall@5"] is not None:
        out["분류로 잃은 Recall@5"] = round(out["분류 없이 검색만 Recall@5"] - out["2단계 종합 Recall@5"], 4)
    return out


def hint_effect(rows):
    # 인사팀: 검색어에 search_hint를 붙였을 때 vs 안 붙였을 때 (힌트가 생긴 문항, 그리고 정답 있는 인사팀 전체)
    a = [r for r in rows if r["set"] == "인사팀" and r["has_gold"]]
    h = [r for r in a if r["search_hint"]]
    on_all = [(HINTED[r["id"]]["r5"] if r["search_hint"] else r["base_r5"]) for r in a]
    return {"힌트 생긴 문항": len(h),
            "힌트 생긴 문항 Recall@5 · 붙임": _mean([r["hint_on_r5"] for r in h]),
            "힌트 생긴 문항 Recall@5 · 안 붙임": _mean([r["hint_off_r5"] for r in h]),
            "인사팀 93 Recall@5 · 붙임(생긴 문항만)": _mean(on_all),
            "인사팀 93 Recall@5 · 전부 안 붙임": _mean([r["base_r5"] for r in a]),
            "힌트로 좋아진 문항": [r["id"] for r in h if r["hint_on_r5"] > r["hint_off_r5"]],
            "힌트로 나빠진 문항": [r["id"] for r in h if r["hint_on_r5"] < r["hint_off_r5"]]}


def speed(rows):
    out = {}
    for name, f in (("임직원", lambda r: r["set"] == "임직원"), ("인사팀", lambda r: r["set"] == "인사팀"), ("전체", lambda r: True)):
        cm = [r["classify_ms"] for r in rows if f(r)]
        sm = [r["base_ms"] for r in rows if f(r) and r["has_gold"]]
        out[name] = {"분류 p50 ms": _pct(cm, 50), "분류 p95 ms": _pct(cm, 95), "분류 문항": len(cm),
                     "검색 p50 ms": _pct(sm, 50), "검색 p95 ms": _pct(sm, 95), "검색 문항": len(sm)}
    return out


def miss_list(rows):
    return [{"id": r["id"], "set": r["set"], "split": r["split"], "질문": r["question"], "예측 의도": r["intent"],
             "이유": r["reason"], "정답 조각": ", ".join(r["gold_chunks"]),
             "정답 조문": ", ".join(CITE_OF_ART.get(a, a) for a in r["gold_articles"])}
            for r in rows if r["has_gold"] and not r["searched"]]


def zero40_list(rows):
    return [{"id": r["id"], "set": r["set"], "split": r["split"], "질문": r["question"], "검색어": r["query"],
             "정답 조각": ", ".join(r["gold_chunks"]), "정답 조문": ", ".join(CITE_OF_ART.get(a, a) for a in r["gold_articles"]),
             "상위 3": " | ".join(r["pipe_top5"][:3])}
            for r in rows if r["has_gold"] and r["searched"] and r["pipe_r40"] == 0]


def make_summary(rows, meta):
    return {"meta": meta,
            "main": {name: group_summary([r for r in rows if f(r)]) for name, f in GROUPS.items()},
            "extra": {name: group_summary([r for r in rows if f(r)]) for name, f in EXTRA_GROUPS.items()},
            "hint_effect": hint_effect(rows), "speed": speed(rows),
            "llm": {"calls": CALLS["n"], "fail": CALLS["fail"], "request_fail": CALLS["fail"] - CALLS["bad_json"],
                    "bad_json": CALLS["bad_json"], "rule_fallback": dict(FALLBACK),
                    "questions_with_fallback": sum(bool(r["fallback"]) for r in rows),
                    "avg_ms": round(float(np.mean(CALLS["ms"])), 1) if CALLS["ms"] else None},
            "search_miss": miss_list(rows), "recall40_zero": zero40_list(rows)}

In [ ]:
ROWS = build_rows()
META = {"commits": COMMITS, "gpu": f"{GPU_NAME} × {N_GPU}" if "GPU_NAME" in globals() else "cpu",
        "classify_llm": CFG["llm_model"], "classify_served_name": CFG["served_name"],
        "embed_model": CFG["embed_model"], "rerank_model": CFG["rerank_model"],
        "search": "B (조문 검색 + 항 리랭크, RAG feat/b-search-notebook 04048ce)", "search_cfg": {k: v for k, v in CFG.items() if k not in ("branches",)},
        "classify_today": CLASSIFY_TODAY, "classify_today_note": "분류 LLM 기준일은 date.today() (코드 안 고침)",
        "run_at": datetime.now().isoformat(timespec="seconds")}
SUMMARY = make_summary(ROWS, META)

MAIN_KEYS = ["문항(정답 조각 있음)", "검색 누락률", "검색 누락", "  └ 기밀문서", "2단계 종합 Recall@5", "분류 없이 검색만 Recall@5",
             "분류로 잃은 Recall@5", "검색한 문항", "검색한 문항 Recall@5", "검색한 문항 MRR", "검색한 문항 nDCG@5", "정답 항 전달률",
             "정답 항 전달률 (auto=false, 따로)"]
TABLE = pd.DataFrame(SUMMARY["main"]).loc[MAIN_KEYS]
display(TABLE)
display(pd.DataFrame(SUMMARY["main"] | SUMMARY["extra"]))
print("속도"); display(pd.DataFrame(SUMMARY["speed"]))
print("힌트 효과 (인사팀)"); display(pd.Series(SUMMARY["hint_effect"]))
print("분류 LLM", SUMMARY["llm"])

### 검색 누락 문항 · 후보 40개에도 정답이 없는 문항

In [ ]:
pd.set_option("display.max_colwidth", 80)
MISS = pd.DataFrame(SUMMARY["search_miss"])
Z40 = pd.DataFrame(SUMMARY["recall40_zero"])
print(f"검색 누락 {len(MISS)}문항 · 이유: {MISS['이유'].value_counts().to_dict() if len(MISS) else {}}")
display(MISS)
print(f"검색했는데 Recall@40 = 0: {len(Z40)}문항")
display(Z40)

## 12. 저장 (rows.jsonl + summary.json → zip · Drive)

In [ ]:
def save_outputs(out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "rows.jsonl").write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in ROWS), encoding="utf-8")
    (out_dir / "summary.json").write_text(json.dumps(SUMMARY, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
    MISS.to_csv(out_dir / "search_miss.csv", index=False, encoding="utf-8-sig")
    Z40.to_csv(out_dir / "recall40_zero.csv", index=False, encoding="utf-8-sig")
    (out_dir / "notion.md").write_text(notion_md(), encoding="utf-8")
    return shutil.make_archive(str(out_dir), "zip", out_dir)


def _f(v):
    return "-" if v is None else (f"{v:.3f}" if isinstance(v, float) else str(v))


def notion_md():
    cols = list(GROUPS)
    keys = [("검색 누락률", "검색 누락률"), ("검색 누락", "검색 누락 (문항)"), ("  └ 기밀문서", "└ 기밀문서"),
            ("2단계 종합 Recall@5", "**2단계 종합 Recall@5**"), ("분류 없이 검색만 Recall@5", "분류 없이 검색만 Recall@5"),
            ("분류로 잃은 Recall@5", "분류로 잃은 만큼"), ("검색한 문항", "검색한 문항"), ("검색한 문항 Recall@5", "검색한 문항 Recall@5"),
            ("검색한 문항 Recall@10", "검색한 문항 Recall@10"), ("검색한 문항 Recall@40(후보)", "검색한 문항 Recall@40(후보)"),
            ("검색한 문항 MRR", "검색한 문항 MRR"), ("검색한 문항 nDCG@5", "검색한 문항 nDCG@5"),
            ("정답 항 전달률", "정답 항 전달률"), ("정답 항 전달률 (auto=false, 따로)", "정답 항 전달률 (auto=false)")]
    m = SUMMARY["main"]
    L = ["| 지표 | " + " | ".join(f"{c} ({m[c]['문항(정답 조각 있음)']})" for c in cols) + " |",
         "| --- |" + " --- |" * len(cols)]
    L += [f"| {label} | " + " | ".join(_f(m[c][k]) for c in cols) + " |" for k, label in keys]
    sp, llm, h = SUMMARY["speed"], SUMMARY["llm"], SUMMARY["hint_effect"]
    L += ["", f"- 불필요한 검색 (정답 조각 없는데 검색): " + ", ".join(f"{c} {m[c]['참고: 정답 조각 없는데 검색']}/{m[c]['참고: 정답 조각 없는 문항']}" for c in cols),
          f"- 인사팀 힌트 효과 (힌트 생긴 {h['힌트 생긴 문항']}문항 Recall@5): 붙임 {_f(h['힌트 생긴 문항 Recall@5 · 붙임'])} vs 안 붙임 {_f(h['힌트 생긴 문항 Recall@5 · 안 붙임'])}",
          f"- 속도: 분류 p50 {_f(sp['전체']['분류 p50 ms'])} · p95 {_f(sp['전체']['분류 p95 ms'])} ms / 검색 p50 {_f(sp['전체']['검색 p50 ms'])} · p95 {_f(sp['전체']['검색 p95 ms'])} ms",
          f"- 분류 LLM 호출 {llm['calls']}번 · 실패 {llm['fail']}번 (요청 {llm['request_fail']} · JSON 아님 {llm['bad_json']}) · 규칙 대체 {llm['rule_fallback'] or 0}",
          f"- 분류 LLM 기준일: date.today() = {META['classify_today']} (코드 안 고침)",
          "- 커밋: " + " / ".join(f"{k} `{v}`" for k, v in META["commits"].items()),
          f"- GPU {META['gpu']} · 분류 {META['classify_llm']} · 검색 {META['embed_model']} + {META['rerank_model']} (B)"]
    return "\n".join(L)


ZIP = save_outputs(OUT_DIR)
print(notion_md())
print("\n저장:", ZIP)

In [ ]:
from google.colab import files
files.download(ZIP)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    dst = Path("/content/drive/MyDrive/hr_chatbot_eval")
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copy(ZIP, dst)
    print("Drive에도 저장:", dst / Path(ZIP).name)
except Exception as e:
    print(f"Drive 연결 안 됨 ({type(e).__name__}) → 내려받은 zip을 노션에 첨부하세요")